**Fase 1: Ingestão de Dados e Enriquecimento de Metadados (Metadata-First)**

* O processamento iniciou com a extração das opiniões a partir do arquivo JSONL original (`PublicHearingBR_LDS.jsonl`).


* Em vez de lidar com as transcrições brutas, focamos nos metadados validados, que já agrupam os envolvidos e os resumos de suas respectivas falas.


* Utilizamos expressões regulares para isolar e padronizar a filiação partidária ou grupo da sociedade civil de cada orador a partir do campo de cargos.

**Fase 2: Mineração de Proposições Locais por Audiência**

* Agrupamos as opiniões limpas pelo ID e assunto de cada audiência e acionamos um Modelo de Linguagem de Grande Escala (LLM) para identificar de 1 a 3 proposições polêmicas centrais que dividiram os debatedores.


* Para contornar erros de formatação e limites da API, implementamos um sistema de extração baseado no prefixo textual `PROP:`.


* **Tecnologias e Modelos utilizados:** Integração da NVIDIA via pacote `langchain_nvidia_ai_endpoints`. Orquestração de instruções de sistema e usuário com `ChatPromptTemplate` do ecossistema LangChain. O modelo testado e utilizado para o raciocínio e extração foi o `nvidia/nemotron-3-nano-30b-a3b`.


**Fase 3: Detecção de Postura (Stance Detection) Paralelizada**

* Para atender ao objetivo de descobrir "quem concorda com quem" (Trilha B do desafio), desdobramos a base para criar pares individuais contendo uma opinião e uma proposição alvo.


* O LLM foi instruído como um juiz imparcial com regras estritas para classificar cada interação exclusivamente como "A FAVOR", "CONTRA" ou "NEUTRO".

* **Tecnologias e Modelos utilizados:** Modelo `nvidia/nemotron-3-nano-30b-a3b` configurado com `temperature=0.0` para saídas determinísticas.



**Fase 4: Construção Topológica e Grafo de Alinhamento**

* Utilizamos os dados classificados para montar uma rede complexa onde os nós representam os participantes das audiências (com cores definidas por seus partidos políticos).


* As arestas foram criadas aplicando uma regra matemática simples de concordância: se dois oradores expressaram a mesma postura ("A FAVOR" ou "CONTRA") para uma mesma proposição, a ligação entre eles ganhava +1 de peso; caso divergissem, o peso recebia -1.


* Filtramos as arestas com peso negativo ou neutro e exportamos os componentes conexos para um ambiente visual.


* **Tecnologias utilizadas:** Biblioteca `networkx` para a modelagem estrutural e aplicação da Teoria dos Grafos. Função `combinations` do módulo `itertools` para realizar o cruzamento ideológico de todos contra todos em cada pauta. Biblioteca `pyvis.network` para renderizar o grafo em um dashboard HTML com física de repulsão (`spring_strength` e `central_gravity`).

In [1]:
import os
NVIDIA_API_KEY = os.environ.get("NVIDIA_API_KEY", "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")

In [2]:
import json
import pandas as pd
import re

# Caminho para o seu arquivo do dataset
caminho_arquivo = 'PublicHearingBR_LDS.jsonl'

registros_opinioes = []

# 1. Leitura e "Desdobramento" (Unrolling) do JSONL
with open(caminho_arquivo, 'r', encoding='utf-8') as f:
    for linha in f:
        amostra = json.loads(linha)
        id_audiencia = amostra.get('id')
        
        # O assunto global daquela audiência
        assunto_audiencia = amostra.get('metadados', {}).get('assunto', '')
        
        # Iterando sobre os participantes
        for envolvido in amostra.get('metadados', {}).get('envolvidos', []):
            nome = envolvido.get('nome', '').strip().title()
            cargo = envolvido.get('cargo', '').strip()
            
            # Iterando sobre cada opinião individual (O "Filé Mignon" do dataset)
            for opiniao in envolvido.get('opinioes', []):
                registros_opinioes.append({
                    'id_audiencia': id_audiencia,
                    'assunto_audiencia': assunto_audiencia,
                    'nome': nome,
                    'cargo': cargo,
                    'opiniao_limpa': opiniao.strip()
                })

df_base = pd.DataFrame(registros_opinioes)

# 2. Enriquecimento: Extração de Filiação Partidária (Preparação para os Grafos)
# Captura o partido de Deputados e Senadores (ex: "Deputado (PT-SP)")
regex_politicos = r"(?:Deputad[ao]|Senador[a]?)\s*\(\s*(?P<partido>[^-]+?)\s*-\s*[A-Za-z]{2}\s*\)"
partidos_extraidos = df_base['cargo'].str.extract(regex_politicos, flags=re.IGNORECASE)

# Atribui o partido extraído ou marca como "SOCIEDADE/OUTROS" para não-políticos
df_base['filiacao_rede'] = partidos_extraidos['partido'].str.upper().str.strip()
df_base['filiacao_rede'] = df_base['filiacao_rede'].fillna('SOCIEDADE/OUTROS')

# 3. Limpeza Final
# Removemos opiniões vazias que possam ter vindo do JSON
df_base = df_base[df_base['opiniao_limpa'].astype(bool)]

print(f"Total de opiniões extraídas e validadas: {len(df_base)}")
print("\nAmostra da nova Base de Ouro:")
print(df_base[['nome', 'filiacao_rede', 'opiniao_limpa']].head())

Total de opiniões extraídas e validadas: 2203

Amostra da nova Base de Ouro:
                    nome     filiacao_rede  \
0  Michael Shellenberger  SOCIEDADE/OUTROS   
1  Michael Shellenberger  SOCIEDADE/OUTROS   
2  Michael Shellenberger  SOCIEDADE/OUTROS   
3        Glenn Greenwald  SOCIEDADE/OUTROS   
4        Glenn Greenwald  SOCIEDADE/OUTROS   

                                       opiniao_limpa  
0  Acusou Alexandre de Moraes de censura ao solic...  
1  Divulgou, no início do mês, um compilado de e-...  
2  Defendeu a liberdade de expressão de forma amp...  
3  Acredita que Alexandre de Moraes agiu sem base...  
4  Afirmou que as pessoas não receberam aviso pré...  


In [3]:
import os
import time
import pandas as pd
from langchain_core.prompts import ChatPromptTemplate
from langchain_nvidia_ai_endpoints import ChatNVIDIA

# ---------------------------------------------------------
# DESBLOQUEIO DO JUPYTER (Mostra o texto completo no DataFrame)
# ---------------------------------------------------------
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', None)

# 1. Preparação dos Dados
df_audiencias_agrupadas = df_base.groupby(['id_audiencia', 'assunto_audiencia'])['opiniao_limpa'].apply(
    lambda x: "\n- " + "\n- ".join(x)
).reset_index()

# 2. Configuração do LLM
# Dica: O modelo "nano" é muito pequeno e costuma falhar em seguir regras estritas. 
# Se tiver acesso, tente usar o "nvidia/nemotron-4-340b-instruct" ou volte para o "lightning"
llm_nvidia = ChatNVIDIA(
    model="nvidia/nemotron-3-nano-30b-a3b", 
    api_key=NVIDIA_API_KEY, 
    temperature=0.0, 
    max_completion_tokens=500, # Aumentado para garantir que ele não corte a frase no meio
    timeout=120 
)

# 3. Prompt de Sistema
prompt_proposicao = ChatPromptTemplate.from_messages([
    ("system", """Você é um cientista político analisando debates.
Sua tarefa é extrair de 1 a 3 PROPOSIÇÕES CENTRAIS (afirmações polêmicas ou teses principais) que dividiram os participantes.

REGRAS ABSOLUTAS:
1. Escreva cada proposição em português (PT-BR) de forma direta e assertiva.
2. Inicie OBRIGATORIAMENTE cada proposição com o prefixo exato "PROP: ".
3. NÃO escreva introduções, saudações ou explicações. Retorne apenas as linhas com o prefixo.

Exemplo de saída:
PROP: O saque-aniversário do FGTS deve ser substituído por empréstimo consignado.
PROP: A regulação atual prejudica a inovação no setor.
"""),
    ("human", """Assunto da audiência: "{assunto}"

Opiniões dos participantes:
{opinioes}""")
])

chain_proposicao = prompt_proposicao | llm_nvidia

# 4. Pipeline de Processamento Super-Resiliente com Debug
dicionario_proposicoes = {}
print("Iniciando a mineração de proposições (Modo Debug Ativado)...\n")

for _, row in df_audiencias_agrupadas.iterrows():
    id_aud = row['id_audiencia']
    assunto = row['assunto_audiencia']
    opinioes = row['opiniao_limpa']
    
    print(f"Processando Audiência {id_aud} | Assunto: {assunto}...")
    
    max_tentativas = 4
    for tentativa in range(max_tentativas):
        try:
            resposta = chain_proposicao.invoke({
                "assunto": assunto,
                "opinioes": opinioes
            })
            
            linhas = resposta.content.strip().split('\n')
            proposicoes_limpas = [l.replace("PROP:", "").strip() for l in linhas if l.startswith("PROP:")]
            
            # SE FALHAR, MOSTRA O TEXTO BRUTO PARA VOCÊ LER
            if not proposicoes_limpas:
                print(f"\n  [DEBUG - SAÍDA BRUTA DO MODELO]:\n  {resposta.content}\n")
                raise ValueError("O modelo não gerou o prefixo 'PROP:'.")
            
            dicionario_proposicoes[id_aud] = proposicoes_limpas
            
            for p in proposicoes_limpas:
                print(f"{p}")
            print("-" * 60)
            break 
            
        except Exception as e:
            tempo_espera = 15 * (2 ** tentativa)
            print(f"  [Aviso] Falha na tentativa {tentativa + 1}: {str(e)}...")
            
            if tentativa < max_tentativas - 1:
                print(f"Aguardando {tempo_espera}s antes de tentar novamente...")
                time.sleep(tempo_espera)
            else:
                print(f"Desistindo da audiência {id_aud}.\n")
                dicionario_proposicoes[id_aud] = ["Erro: Falha persistente na API."]

# 5. Mesclando as proposições de volta ao DataFrame Original
df_props = pd.DataFrame([
    {"id_audiencia": k, "proposicoes_audiencia": " | ".join(v)} 
    for k, v in dicionario_proposicoes.items()
])

df_fase2 = pd.merge(df_base, df_props, on="id_audiencia", how="inner")
print("\nAmostra do DataFrame (Visão Completa):")
print(df_fase2[['nome', 'opiniao_limpa', 'proposicoes_audiencia']].head())

/home/nuneslima/.conda/envs/voxai/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Iniciando a mineração de proposições (Modo Debug Ativado)...

Processando Audiência 1 | Assunto: Acusações de censura contra Alexandre de Moraes por exigir bloqueio de contas na rede social X...
Alexandre de Moraes cometeu censura ao exigir bloqueio de contas na rede social X sem aviso prévio, explicações ou devido processo legal.
Existe um “processo industrial de censura” e perseguição covarde contra desafetos do ministro, caracterizando censura prévia e violação de liberdades.
As decisões de remoção de conteúdo são genéricas, sigilosas e não obedecem ao Marco Civil da Internet, carecendo de fundamentação judicial adequada.
------------------------------------------------------------
Processando Audiência 2 | Assunto: Envio de projetos de lei para diminuir custos da energia para o consumidor e tratar da transição energética no setor de transportes...
A privatização da Eletrobras representa uma ameaça à soberania e aos direitos dos consumidores brasileiros.
A falta de transparência sob

In [6]:
import pandas as pd
import time
import os
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
from langchain_core.prompts import ChatPromptTemplate
from langchain_nvidia_ai_endpoints import ChatNVIDIA

# 1. Preparação da Base Completa (Fase 3)
df_fase3 = df_fase2.copy()
df_fase3['proposicao_alvo'] = df_fase3['proposicoes_audiencia'].apply(
    lambda x: [p.strip() for p in str(x).split('|') if p.strip()]
)
df_fase3 = df_fase3.explode('proposicao_alvo').reset_index(drop=True)

print(f"Iniciando Stance Detection em {len(df_fase3)} conexões (Modo Paralelo)...\n")

# 2. Prompt e Configuração
prompt_postura = ChatPromptTemplate.from_messages([
    ("system", """Você é um juiz imparcial analisando debates políticos.
Sua tarefa é ler a OPINIÃO de um participante e compará-la com uma PROPOSIÇÃO central.

A opinião apoia (concorda com), ataca (discorda de) ou é neutra em relação a essa proposição exata?

REGRAS ABSOLUTAS:
1. Responda APENAS com uma destas três palavras: A FAVOR, CONTRA ou NEUTRO.
2. NÃO escreva NADA além disso. Nenhuma pontuação, nenhuma explicação e nenhum raciocínio interno."""),
    ("human", """PROPOSIÇÃO ALVO: "{proposicao}"\n\nOPINIÃO DO ORADOR: "{opiniao}"\n\nPOSTURA:""")
])

chain_postura = prompt_postura | llm_nvidia

# 3. Função de Classificação Ajustada para Multithreading
# Agora ela recebe o índice (idx) para sabermos em qual linha salvar a resposta quando ela voltar fora de ordem
def processar_postura(idx, opiniao, proposicao):
    for tentativa in range(3):
        try:
            resposta = chain_postura.invoke({"proposicao": proposicao, "opiniao": opiniao})
            postura = resposta.content.strip().upper()
            
            if "FAVOR" in postura: return idx, "A FAVOR"
            elif "CONTRA" in postura: return idx, "CONTRA"
            else: return idx, "NEUTRO"
            
        except Exception:
            if tentativa < 2: time.sleep(4)
            else: return idx, "ERRO API"

# 4. Execução Paralela com ThreadPoolExecutor
resultados_postura = {}
MAX_THREADS = 10 # Processa 5 requisições simultâneas para não derrubar a API da NVIDIA

with ThreadPoolExecutor(max_workers=MAX_THREADS) as executor:
    # Submete todas as tarefas para a fila
    futuros = {
        executor.submit(processar_postura, idx, row['opiniao_limpa'], row['proposicao_alvo']): idx
        for idx, row in df_fase3.iterrows()
    }
    
    # tqdm acompanha o progresso conforme as threads vão terminando
    for futuro in tqdm(as_completed(futuros), total=len(df_fase3), desc="Classificando Posturas"):
        idx, postura = futuro.result()
        resultados_postura[idx] = postura

# Mapeia os resultados de volta para o DataFrame na ordem correta
df_fase3['postura_final'] = df_fase3.index.map(resultados_postura)

# Salvamos o checkpoint
arquivo_saida = 'base_posturas_classificadas.csv'
df_fase3.to_csv(arquivo_saida, index=False)
print(f"\n✅ Processamento paralelo concluído e salvo em '{arquivo_saida}'\n")

# 5. ANÁLISES ESTATÍSTICAS

print("=" * 80)
print("1. TOP 10 PROPOSIÇÕES COM MAIS OPINIÕES")
print("-" * 80)
print(df_fase3['proposicao_alvo'].value_counts().head(10))

print("\n" + "=" * 80)
print("2. TOP 10 ORADORES MAIS ATIVOS (Mais opiniões emitidas)")
print("-" * 80)
print(df_fase3['nome'].value_counts().head(10))

print("\n" + "=" * 80)
print("3. VOLUME DE DEBATE POR GRUPO/PARTIDO")
print("-" * 80)
print(df_fase3['filiacao_rede'].value_counts())

print("\n" + "=" * 80)
print("4. MATRIZ DE POLARIZAÇÃO GERAL (Partido x Postura)")
print("-" * 80)
matriz_polarizacao = pd.crosstab(
    index=df_fase3['filiacao_rede'], 
    columns=df_fase3['postura_final'],
    margins=True,
    margins_name="TOTAL_GERAL"
)
# Ordena pelos grupos que mais falaram
matriz_polarizacao = matriz_polarizacao.sort_values(by="TOTAL_GERAL", ascending=False)
print(matriz_polarizacao)

Iniciando Stance Detection em 5959 conexões (Modo Paralelo)...



Classificando Posturas: 100%|██████████| 5959/5959 [1:11:48<00:00,  1.38it/s]



✅ Processamento paralelo concluído e salvo em 'base_posturas_classificadas.csv'

1. TOP 10 PROPOSIÇÕES COM MAIS OPINIÕES
--------------------------------------------------------------------------------
proposicao_alvo
O Ministério da Justiça deve ampliar o financiamento e os recursos do Programa de Proteção Integrada de Fronteiras e do Guardiões da Fronteira para garantir a segurança nas 16.885 km de fronteira brasileira.                   31
É imprescindível rever a legislação para permitir a gratificação e a remuneração dos policiais estaduais que atuam nas fronteiras, bem como destinar 30 % dos recursos do Fundo Nacional de Segurança Pública a esses estados.    31
É necessário criar um marco legal que facil                                                                                                                                                                                       31
A                                                                                             

In [8]:
import pandas as pd
import networkx as nx
from itertools import combinations
from pyvis.network import Network

# 1. Carregando a Base de Ouro Validada
df_resultados = pd.read_csv('base_posturas_classificadas.csv')

# Filtramos apenas os ERROS de API. Mantemos A FAVOR, CONTRA e NEUTRO.
df_valido = df_resultados[df_resultados['postura_final'].isin(['A FAVOR', 'CONTRA', 'NEUTRO'])].copy()

# 2. Inicializando o Grafo
G = nx.Graph()

# 3. Adicionando os Nós (Pessoas) com as Cores dos Partidos
cores_partidos = {
    'PT': '#CC0000',
    'PL': '#222299',
    'NOVO': '#FF6600',
    'PSOL': '#FFCC00',
    'MDB': '#009933',
    'SOCIEDADE/OUTROS': '#888888'
}

for _, row in df_valido.iterrows():
    nome = row['nome']
    partido = row['filiacao_rede']
    cor = cores_partidos.get(partido, '#AAAAAA')
    
    # Adiciona o rótulo visível e o texto de hover (title)
    G.add_node(nome, group=partido, color=cor, label=f"{nome}\n({partido})", title=f"{nome} - {partido}", size=15)

# 4. Calculando as Arestas (O Histórico de Encontros)
for prop, group in df_valido.groupby('proposicao_alvo'):
    oradores = group['nome'].tolist()
    posturas = group['postura_final'].tolist()
    
    # Compara o posicionamento de todos que falaram sobre a mesma proposição
    for i, j in combinations(range(len(oradores)), 2):
        orador1 = oradores[i]
        orador2 = oradores[j]
        
        if orador1 == orador2:
            continue
            
        p1 = posturas[i]
        p2 = posturas[j]
        
        # Categoriza a interação
        if p1 == 'NEUTRO' or p2 == 'NEUTRO':
            tipo_interacao = 'neutro'
        elif p1 == p2:
            tipo_interacao = 'concordancia'
        else:
            tipo_interacao = 'discordancia'
        
        # Cria ou atualiza os contadores da aresta
        if G.has_edge(orador1, orador2):
            G[orador1][orador2][tipo_interacao] += 1
        else:
            G.add_edge(orador1, orador2, concordancia=0, discordancia=0, neutro=0)
            G[orador1][orador2][tipo_interacao] += 1

# 5. Classificação Final e Renderização Visual das Arestas
for u, v, data in G.edges(data=True):
    c = data['concordancia']
    d = data['discordancia']
    n = data['neutro']
    
    # Define a cor e a espessura (value) com base no tipo de interação dominante
    if c > d and c >= n:
        data['color'] = "rgba(0, 0, 255, 0.4)" # Azul translúcido
        data['value'] = c
        data['title'] = f"Aliados (Concordaram: {c} | Discordaram: {d} | Neutros: {n})"
    elif d > c and d >= n:
        data['color'] = "rgba(255, 0, 0, 0.4)" # Vermelho translúcido
        data['value'] = d
        data['title'] = f"Antagonistas (Discordaram: {d} | Concordaram: {c} | Neutros: {n})"
    else:
        data['color'] = "rgba(169, 169, 169, 0.2)" # Cinza bem transparente para não poluir
        data['value'] = n if n > 0 else 1
        data['title'] = f"Contato Neutro (Neutros: {n} | Concordaram: {c} | Discordaram: {d})"

# Removemos nós isolados (pessoas que não cruzaram pautas com ninguém)
G.remove_nodes_from(list(nx.isolates(G)))

print(f"Grafo construído: {G.number_of_nodes()} Oradores conectados por {G.number_of_edges()} relações cruzadas.")

# 6. Exportando para o Dashboard Interativo (Pyvis)
rede_interativa = Network(height="800px", width="100%", bgcolor="#ffffff", font_color="black")
rede_interativa.from_nx(G)

# Ajuste fino da física para suportar a rede mais densa com os neutros
rede_interativa.repulsion(node_distance=180, central_gravity=0.15, spring_length=250, spring_strength=0.05, damping=0.09)

arquivo_dashboard = "dashboard_polarizacao_completa.html"
rede_interativa.show(arquivo_dashboard, notebook=False)

print(f"✅ Dashboard gerado com sucesso! Abra '{arquivo_dashboard}' para visualizar as interações azuis, vermelhas e cinzas.")

Grafo construído: 562 Oradores conectados por 1336 relações cruzadas.
dashboard_polarizacao_completa.html
✅ Dashboard gerado com sucesso! Abra 'dashboard_polarizacao_completa.html' para visualizar as interações azuis, vermelhas e cinzas.
